#### 17 - Model Registration

##### 1. Notebook Objective


This notebook registers the selected fine-tuned ResNet18 classifier in the
Unity Catalog Model Registry.

The model has already been:

- trained;
- selected using validation loss;
- persisted as an MLflow model artifact;
- evaluated on the test dataset;
- compared with the Custom CNN;
- analyzed using Explainable AI;
- assessed using Responsible AI principles.

Therefore, this notebook does not retrain the model.

The goals are to understand:

1. Why model registration is different from MLflow experiment tracking.
2. The relationship between a run, model artifact, registered model, and
   model version.
3. How to register an existing MLflow model in Unity Catalog.
4. How model versions provide traceability and governance.
5. How to retrieve and validate a registered model.
6. How registration prepares the model for deployment.

##### 2. Where are we in the lifecycle?

``` text

Images
  ↓
Preprocessing
  ↓
Custom CNN
  ↓
Transfer Learning
  ↓
Fine-tuned ResNet18
  ↓
Evaluation
  ↓
XAI
  ↓
RAI
  ↓
MLflow Tracking
  ↓
★ MODEL REGISTRATION ★
  ↓
Deployment
  ↓
Endpoint Testing

```

Our selected model currently exists here:

``` text

MLflow Run
3f9b44ae55a449c5850f0b48c4ee4fa6
        │
        └── model
              ↓
runs:/3f9b44ae55a449c5850f0b48c4ee4fa6/model

```

That's a persisted model artifact.

Now we'll create:

``` text

Unity Catalog

dbw_agentic_ai_dev
        ↓
product_defect_ai
        ↓
product_defect_classifier
        ↓
Version 1

```

##### 3. Tracking versus Registration

**MLflow Tracking**

We currently have:

``` text

Experiment
    ↓
Run
    ↓
parameters
metrics
artifacts
model

```

This answers: What happened during development?

**Model Registry**

Registration creates:

``` text

Registered Model
       ↓
Version 1
Version 2
Version 3
...

```
This answers: Which versions of this model are being formally managed?

So:

``` text


TRAINING HISTORY                 MODEL LIFECYCLE

MLflow Experiment               Registered Model
       ↓                              ↓
MLflow Run                       Version 1
       ↓                         Version 2
Model Artifact                   Version 3

```

They are connected, but not the same thing.

##### 4. Why do we need model versions?

Imagine that six months later we retrain using more production data.

Maybe:

``` text

Version 1
ResNet18
current dataset

Version 2
ResNet18
+ new casting images

Version 3
new architecture
+ more production data

```
We don't want:

- model_final
- model_new
- model_new_final
- model_final_latest

Instead:

``` text
product_defect_classifier
│
├── Version 1
├── Version 2
└── Version 3
```
Each version remains traceable to its source model.

##### 5. Why register ResNet18 rather than the Custom CNN?

We evaluated both models before making this decision.

Our measured results were approximately:

- Custom CNN : Original test accuracy: 83.22%
- ResNet18: Original test accuracy: 99.58%

After removing exact development/test duplicates:

- Custom CNN: 83.26%
- ResNet18:   99.54%

The ResNet also had only three missed defects on the original test set compared with 68 for the custom CNN.

Therefore the selected deployment candidate from our model-development process is the fine-tuned ResNet18.

Importantly, this does not mean: Custom CNN disappears

Its MLflow run remains part of the development history.

We simply don't need to register every experiment as a deployment candidate.

##### 6. Notebook Imports

In [0]:
import mlflow
from mlflow import MlflowClient

##### 7. Configuration

In [0]:
from src.project_config import (
    REGISTERED_MODEL_NAME,
)

In [0]:
RESNET18_RUN_ID = (
    "3f9b44ae55a449c5850f0b48c4ee4fa6"
)

RESNET18_MODEL_URI = (
    f"runs:/{RESNET18_RUN_ID}/model"
)

print(
    "Source model:",
    RESNET18_MODEL_URI,
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

##### 8. Tell MLflow to use Unity Catalog

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

MLflow needs to know which registry we're targeting.

Conceptually:

``` text

MLflow
  ↓
Where should I register this model?

"databricks-uc"
  ↓
Unity Catalog Model Registry

```

In [0]:
print(
    "Registry URI:",
    mlflow.get_registry_uri(),
)

##### 8A. Load the existing trained model

In [0]:
import torch

model = mlflow.pytorch.load_model(
    RESNET18_MODEL_URI
)

model.eval()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print("Device:", device)
print(model.fc)

##### 8B. Recreate the correct preprocessing pipeline

In [0]:
#Because the signature must represent the actual model input, use our existing production module:

from src.data_preparation import (
    create_image_dataloaders,
)

data = create_image_dataloaders(
    preprocessing_mode="pretrained"
)

In [0]:
#Now get one real preprocessed image:
example_images, _ = next(
    iter(data.val_loader)
)

example_input = (
    example_images[:1]
    .to(device)
)

print(
    "Example input shape:",
    example_input.shape,
)

##### 8C. Generate the corresponding output

In [0]:
with torch.no_grad():

    example_output = model(
        example_input
    )

print(
    "Example output shape:",
    example_output.shape,
)

INPUT
[batch, 3, 224, 224]

        ↓

     ResNet18

        ↓

OUTPUT
[batch, 2]
logits

##### 8D. Infer the signature

In [0]:
from mlflow.models import infer_signature

#Convert the tensors to NumPy:

example_input_numpy = (
    example_input
    .detach()
    .cpu()
    .numpy()
)

example_output_numpy = (
    example_output
    .detach()
    .cpu()
    .numpy()
)

In [0]:
signature = infer_signature(
    example_input_numpy,
    example_output_numpy,
)

print(signature)

##### 8E. Log the SAME model with the signature

We don't want to modify the historical training run.

Instead, create a new packaging run.

That's cleaner from a governance perspective:

``` text

Original training run
3f9b44...
     ↓
historical training evidence
     ↓
load selected model
     ↓
Packaging run
     ↓
same weights
+ signature
+ input example
     ↓
Unity Catalog

```

In [0]:
if mlflow.active_run() is not None:
    mlflow.end_run()

packaging_run = mlflow.start_run(
    run_name="resnet18_registration_package"
)

packaging_run_id = (
    packaging_run.info.run_id
)

print(
    "Packaging Run ID:",
    packaging_run_id,
)

In [0]:
#Log lineage back to the original training run:
mlflow.set_tags(
    {
        "model_type": "resnet18",
        "purpose": "unity_catalog_registration",
        "source_training_run_id": (
            RESNET18_RUN_ID
        ),
    }
)

In [0]:
#Now log the model:

model_info = mlflow.pytorch.log_model(
    pytorch_model=model,
    name="model",
    signature=signature,
    input_example=example_input_numpy,
)

In [0]:
print(
    "Logged model URI:",
    model_info.model_uri,
)

In [0]:
mlflow.end_run()

##### 8F. Register the newly packaged model

In [0]:
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=REGISTERED_MODEL_NAME,
)

In [0]:
print(
    "Registered model:",
    registered_model.name,
)

print(
    "Version:",
    registered_model.version,
)

print(
    "Status:",
    registered_model.status,
)

``` text

Original training run
3f9b44...
        │
        │ model weights unchanged
        ↓
Registration packaging run
        │
        ├── same ResNet18
        ├── signature
        ├── input example
        └── source-training-run tag
                 ↓
         Unity Catalog
                 ↓
product_defect_classifier
                 ↓
            Model Version

```

Are we creating a second model?

This is a subtle distinction.

We are creating a new MLflow model package, but we are not training a new neural network.

The underlying:

- architecture
- weights
- learned features
- predictions

remain the same.

What we're adding is the missing production metadata:
``` text

BEFORE

trained model
+
weights


AFTER

same trained model
+
same weights
+
input schema
+
output schema
+
input example
+
lineage metadata

```

So this is model packaging, not model retraining.

In [0]:
version = registered_model.version

print("Registered version:", version)

| Identity               | Your value                     | Meaning                                                                       |
| ---------------------- | ------------------------------ | ----------------------------------------------------------------------------- |
| **Training Run ID**    | `3f9b44ae...`                  | Execution where ResNet actually learned its weights                           |
| **Packaging Run ID**   | `981b01f8...`                  | Execution where the trained model was repackaged with signature/input example |
| **Logged Model ID**    | `m-a2d472...`                  | MLflow model package produced by packaging                                    |
| **Registered Version** | `product_defect_classifier` v1 | Governed Unity Catalog version                                                |


**Run IDs identify executions; the m-... ID identifies a logged model package; Version 1 identifies the governed Unity Catalog model version.**

``` text
TRAINING RUN
3f9b44...
"Learn the weights"
       │
       ↓
Fine-tuned ResNet18
       │
       │ load same model
       ↓
PACKAGING RUN
981b01...
"Prepare for registration"
       │
       ├── same weights
       ├── signature
       └── input example
       │
       ↓
LOGGED MODEL
m-a2d472...
"Packaged MLflow model"
       │
       ↓
UNITY CATALOG
product_defect_classifier
Version 1
"Governed model version"

```

In [0]:
model_description = """
Fine-tuned ResNet18 classifier for casting-product quality inspection.

Classes:
- def_front
- ok_front

Training strategy:
- ImageNet-pretrained ResNet18
- feature extraction
- partial fine-tuning of layer4

Model selection:
- lowest validation loss

Evaluation:
- approximately 99.58% accuracy on the original 715-image test split
- approximately 99.54% accuracy on the 651-image non-duplicate test subset
- three defect false negatives observed

Responsible AI considerations:
- exact development/test duplicates were identified during evaluation
- high-confidence incorrect predictions can occur
- production robustness across cameras, lighting conditions, manufacturing
  equipment, and distribution shift has not been established

Explainability:
- Grad-CAM provides diagnostic class-influence information
- Grad-CAM should not be interpreted as verified defect localization
""".strip()

client = MlflowClient()

client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=version,
    description=model_description,
)

##### 18. Add structured tags

In [0]:
version_tags = {
    "task": "image_classification",
    "architecture": "resnet18",
    "training_strategy": (
        "feature_extraction_then_layer4_fine_tuning"
    ),
    "classes": "def_front,ok_front",
    "source_training_run_id": RESNET18_RUN_ID,
    "validation_status": (
        "experimental_evaluation_complete"
    ),
}

for key, value in version_tags.items():

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=version,
        key=key,
        value=value,
    )

##### 19. Verify the lineage

In [0]:
model_version = client.get_model_version(
    name=REGISTERED_MODEL_NAME,
    version=version,
)

print(
    "Registered model:",
    model_version.name,
)

print(
    "Version:",
    model_version.version,
)

print(
    "Status:",
    model_version.status,
)

print(
    "Source:",
    model_version.source,
)

print(
    "Run ID:",
    model_version.run_id,
)

print(
    "\nDescription:"
)

print(
    model_version.description
)

In [0]:
##### 20. Load Version 1 from Unity Catalog

In [0]:
registered_model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}/{version}"
)

print(
    "Registered Model URI:",
    registered_model_uri,
)

In [0]:
registered_resnet = (
    mlflow.pytorch.load_model(
        registered_model_uri
    )
)

registered_resnet.eval()

print(
    registered_resnet.fc
)

##### 21. What did READY mean?

That means the model-version registration operation completed successfully and the registered version is available.

It does not mean: endpoint is running

and it does not independently establish: production safety approved

Those are different lifecycle concepts.

So our current state is:
``` text

Fine-tuned ResNet18
        ✓

Test evaluation
        ✓

XAI
        ✓

RAI assessment
        ✓

MLflow tracking
        ✓

Signature + input example
        ✓

Unity Catalog registration
        ✓

product_defect_classifier
Version 1
Status READY
        ✓

        ↓

Serving endpoint
        NOT YET

```

**Why I prefer a new packaging run**

Technically, we could try to manipulate the old development history, but I don't recommend that.

The historical training run should continue to mean: This is what happened when we trained the model.

The new run means: This is how we packaged that selected model for registration.

That gives us much cleaner lineage:

``` text

TRAINING RUN
"What produced the weights?"

       ↓

PACKAGING RUN
"How was the selected model prepared
for registry/deployment?"

       ↓

REGISTERED VERSION
"Which governed model are we using?"

```

That's actually stronger MLOps practice than pretending the signature existed when the model was originally trained.

We previously said we didn't need to relog the model just to learn signatures. That was true for loading/evaluating the model, but your current Databricks/Unity Catalog environment makes a signature mandatory for registration.

``` text

So our updated understanding is:

Loading existing MLflow model
        ↓
signature optional

Unity Catalog registration
        ↓
signature required

```

##### 10. Important: registration does not retrain or copy our Python training process

When we call: mlflow.register_model(...)

MLflow is not doing:

``` text

load images
↓
train ResNet
↓
fine-tune
↓
evaluate

```
It is taking the already-persisted model: MLflow model artifact

and creating a managed model version referencing that model artifact.

So:
``` text

TRAINING
expensive
already finished

REGISTRATION
lifecycle/governance operation

```

##### 11. What exactly did we register?

We registered:

fine-tuned ResNet18 model

We did not register:

training notebook
Grad-CAM notebook
dataset
DataLoader
optimizer
training loop

The registered model represents the model artifact.

But its lineage connects us back to the source MLflow run.

Conceptually:

Registered Model Version
          ↓
Source Model Artifact
          ↓
MLflow Run
          ↓
Parameters
Metrics
Training evidence

That's why MLflow tracking before registration is valuable.